## I2 reader mode: CONSTRUCTED monthly factor attribution

All newly computed values are **CONSTRUCTED method demonstrations**, never market observations or historical verification. Historical tables are separately labelled replay. French raw/extracted/processed bytes are not bundled. French route: A10-A limited-scope design accepted, execution blocked; A10-A.1 protocol technically accepted pending permissions and execution authorization. G01 rights/subject/use unknown; G05 A10-B not authorized.

Original explanatory prose below describes the course method. The reader-mode code and labels define which portions execute here. No P6 retraining or Ch39 historical row reconstruction.


# Project 2：CAPM 与 Fama–French 因子回归

本 Notebook 对三个真实月度行业组合（NoDur、HiTec、Enrgy）分别估计 CAPM、FF3、FF5，共九组模型。窗口固定为 2010-01 至 2019-12，每个对象 120 个月。输入为 2026-10-05 冻结的 French 官方快照。

目标是学习历史因子归因：资产收益与发布因子的条件关系、模型截距、解释方差及残差如何随基准模型改变。结果是关联，不是因果效应；截距是模型相关历史 Alpha，不代表未来超额收益或可交易优势。

本项目使用资本资产定价模型（Capital Asset Pricing Model, CAPM）、Fama–French 三因子模型（Fama–French Three-Factor Model, FF3）与五因子模型（Fama–French Five-Factor Model, FF5）。普通最小二乘法（Ordinary Least Squares, OLS）估计系数；异方差与自相关一致协方差估计（Heteroskedasticity and Autocorrelation Consistent Covariance Estimation, HAC）用于推断；置信区间（Confidence Interval, CI）与 Holm 逐步多重检验校正（Holm Step-Down Multiple-Testing Adjustment）分别描述单项估计不确定性与所列检验族的校正。原理与手算见第8、15章。


## 方程、单位和时间

对资产 \(i\)，令月度总收益为 \(R_{i,t}\)，发布的一月期国库券代理为 \(RF_t\)，超额收益为 \(y_{i,t}=R_{i,t}-RF_t\)。拟合

\[
y_{i,t}=\alpha_i+\boldsymbol\beta_i^{\top}\mathbf f_t+\varepsilon_{i,t}.
\]

- CAPM 用 FF3 文件的 `Mkt-RF`；FF3 与 FF5 使用各自官方发布的因子序列。
- `Mkt-RF`、`SMB`、`HML`、`RMW`、`CMA` 已是因子价差，原样进入模型。只从行业总收益减一次同月 `RF`。
- 月收益及价差在处理文件中是小数；不再除以 100。`2010-01` 是收益区间，不是交易日或月内的可得时间。
- HAC 主结果用 Bartlett 核和 6 个按月观测行计的滞后；另输出 3、6、12 阶全部系数推断。置信区间和 p 值依赖未由代码验证的假设。
- 时间切分把 2010–2014 系数应用到 2015–2019 同期已实现因子，仅作系数稳定性与归因误差描述。评价月因子在月末后才完整可知，故不是事前预测或策略测试。

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("project02_factor_regression_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from quant_course.factor_models import fit_monthly_factor_model
print("CONSTRUCTED factor attribution; French snapshot/alignment verification SKIPPED: rights unknown")


In [ ]:
rng = np.random.default_rng(102)
idx = pd.period_range("2010-01", periods=120, freq="M", name="month")
factors = pd.DataFrame(rng.normal(0, 0.03, (120,5)), index=idx, columns=["Mkt-RF","SMB","HML","RMW","CMA"])
factors["RF"] = 0.001
asset = factors["RF"] + 0.002 + 1.1*factors["Mkt-RF"] + 0.4*factors["SMB"] + rng.normal(0,0.015,120)
models = {"CAPM": ("Mkt-RF",), "FF3": ("Mkt-RF","SMB","HML"), "FF5": ("Mkt-RF","SMB","HML","RMW","CMA")}
fits = {m:fit_monthly_factor_model(asset,factors,cols,hac_maxlags=6) for m,cols in models.items()}
assert all(f.nobs == 120 for f in fits.values())


## 九组模型结果

表中 Alpha 与区间以月度百分点表示，Beta 为无量纲条件载荷。表中保留负值和接近零的结果；`NaN` 的其他因子行表示该模型没有估计该系数。HAC 95% 区间按六阶带宽计算，是逐项区间，不是经 Holm 调整的同时置信区间。九个 Alpha 的 Holm 调整 p 值作为同一展示检验族的补充；其他模型搜索并未纳入这个校正。

In [ ]:
for model, fit in fits.items():
    print(model, "CONSTRUCTED OLS + monthly HAC(6)")
    display(fit.coefficient_table)
    fit.coefficient_table.to_csv(RUN_DIR / (model + "_coefficients_CONSTRUCTED.csv"), index=False)


## HAC 带宽和样本方差

改变 HAC 带宽会改变标准误、区间和 p 值，不会改变 OLS 点估计。样本方差分解使用相同 120 个月和 `ddof=1`：\(\operatorname{Var}(y)=\operatorname{Var}(\hat y)+\operatorname{Var}(e)\)。OLS 的样本内投影恒等式不表示未来风险可被预测，也不能将有相关性的因子各自解释成相互独立的方差份额。

In [ ]:
for lag in (3, 6, 12):
    f = fit_monthly_factor_model(asset, factors, models["FF3"], hac_maxlags=lag)
    print("HAC bandwidth", lag, "months; R2", f.r_squared)
    display(f.coefficient_table)
fit = fits["FF3"]
np.testing.assert_allclose(asset - factors["RF"], fit.fitted_excess + fit.residuals)


## 图形阅读

- Alpha 图：横轴为月度截距的百分点，误差线是 HAC 95% 区间；零线表示给定模型的条件截距为零。
- 因子载荷图：分别展示 Mkt-RF、SMB、HML、RMW、CMA 的条件 Beta 和 HAC 区间；遗漏因子的行不估计、不画成零。
- 方差图：每条 100% 堆叠条把样本超额收益方差拆成拟合项和残差项；它是样本内会计分解。
- 残差图：纵轴为实现的同月超额收益减拟合值，单位是月度百分点，横轴以月末作图标签展示区间。
- 时间顺序归因误差图：以 2010–2014 估计系数计算 2015–2019 同期因子归因误差的 RMSE。已实现因子不是提前可知变量。

In [ ]:
fig, ax = plt.subplots(figsize=(8,3))
ax.plot(fits["FF3"].residuals.to_numpy())
ax.set(title="CONSTRUCTED FF3 residuals", xlabel="Month", ylabel="Decimal excess return")
fig.tight_layout(); fig.savefig(RUN_DIR / "project02_residuals_CONSTRUCTED.png", dpi=130)
plt.show()


## 来源与解释边界

官方数据与构造页面：[Kenneth R. French Data Library](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html)、[10 Industry Portfolios](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/det_10_ind_port.html)、[FF3 因子说明](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_factors.html)、[FF5 因子说明](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html)。因子模型的原始研究背景见 [Fama and French (1993)](https://www.sciencedirect.com/science/article/pii/0304405X93900235) 与 [Fama and French (2015)](https://www.sciencedirect.com/science/article/pii/S0304405X14002323)。来源下载时间、CRSP 版本、文件路径与 SHA-256 均记在 `projects/project02/data_manifest.json`。

FF3 与 FF5 的 SMB 官方构造不同：FF3 使用规模×账面市值比分组，FF5 平均账面市值比、盈利、投资三组规模价差；本项目未用底层组合逐月复制差异。Mkt-RF 文件数值差异的具体原因仍未查明。因此不把两组官方回归视为严格嵌套比较。参见[官方FF3说明](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_factors.html)与[官方FF5说明](https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/Data_Library/f-f_5_factors_2x3.html)。HAC 只是推断协方差估计选择，不能消除模型遗漏、历史修订、市场结构变化或有限样本问题。无交易账本，不估成本后 alpha。